In [1]:
from bs4 import BeautifulSoup
from dataclasses import dataclass
from re import search
from requests import Session
from sqlite3 import connect
from tqdm import tqdm
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


# Hàm hỗ trợ: Chuyển đổi mã học kỳ
Ví dụ năm học 2025-2026, học kỳ 1:
- Trong database sẽ lưu `term_id = 251`, với `25` là năm học bắt đầu và `1` là học kỳ.
- Nhưng trên hệ thống tra cứu đăng ký học thì là `term_id = 044`, ta cần hai hàm để encode và decode dữ liệu này.

In [2]:
def term_id_encode(term_id_decoded: str) -> str:
  year = int(term_id_decoded[:2])
  semester = int(term_id_decoded[2])
  term_id_encoded = (year - 10) * 3 + semester - 2
  return str(term_id_encoded).rjust(3, '0')

def term_id_decode(term_id_encoded: str) -> str:
  term_id = int(term_id_encoded)
  semester = (term_id + 1) % 3 + 1
  year = (term_id + 1) // 3 + 10
  return str(year).rjust(2, '0') + str(semester)

# Class lưu thông tin của một record trong bảng dữ liệu đăng ký học

In [3]:
@dataclass
class DkmhUETRecord:
  student_id:     str
  student_name:   str
  birth_date:     str
  student_class:  str
  section_id:     str
  course_name:    str
  section_group:  str
  course_credits: int
  note:           int
  term_id:        str

  @property
  def course_id(self) -> str:
    patterns = [
      r'^[A-Z]{3}\s?\d{4}[#E*]*', # MAT1041, INT 3103, INT3405E, INT3405#, AIT3040**
      r'^[A-Z]{3}\.[A-Z]{2,3}\d{4}' # UET.CS1058, UET.MAT1050
    ]
    for p in patterns:
      if (match := search(p, self.section_id)):
        return match.group()
    return ""

# Hàm parse dữ liệu đăng ký học lấy được từ web
Khi crawl dữ liệu từ web https://daotaodaihoc.uet.vnu.edu.vn/qldt về, ta có được HTML text nên cần viết hàm để parse dữ liệu này.

In [4]:
def parse_html(html_string: str) -> list[DkmhUETRecord]:
  records = []
  for tr in BeautifulSoup(html_string, "html.parser").find_all("tr"):
    if tr.get("class") not in [["even"], ["odd"]]:
      continue
    data = [td.text for td in BeautifulSoup(str(tr), "html.parser").find_all("td")]
    # Kiểm tra lỗi dữ liệu trên web: dòng này thiếu thông tin thì bỏ qua
    if "" in data or "\xa0" in data:
      continue
    day, month, year = data[3].split('/')
    records.append(DkmhUETRecord(
      student_id     = data[1],
      student_name   = data[2],
      birth_date     = f"{year}-{month}-{day}",  # Lưu đúng chuẩn ISO Format
      student_class  = data[4],
      section_id     = data[5],
      course_name    = data[6],
      section_group  = data[7],
      course_credits = int(data[8]),
      note           = data[9],
      term_id        = term_id_decode(data[10]),
    ))
  return records

# Hàm lấy dữ liệu đăng ký học

In [5]:
session = Session()
session.verify = False

def crawl(term_id: str | int) -> list[DkmhUETRecord]:
  records = []
  term_id_encoded = term_id_encode(str(term_id))
  for page in range(1, 10000000):
    response = session.get(
      url="https://daotaodaihoc.uet.vnu.edu.vn/qldt",
      params={
        "pageSize": 25000,
        "SinhvienLmh_page": page,
        "SinhvienLmh[term_id]": term_id_encoded
      }
    )
    summary = search(r'Kết quả từ \d+ tới (\d+) trên (\d+)', response.text[:12000])
    if not summary:
      break
    records.extend(parse_html(response.text))
    if summary.group(1) == summary.group(2):
      break
  return records

# Tạo database

In [6]:
def create_database() -> None:
  with connect("database.db") as conn:
    conn.executescript("""
      DROP TABLE IF EXISTS students;
      CREATE TABLE students (
        student_id     TEXT  PRIMARY KEY,
        student_name   TEXT,
        birth_date     TEXT,
        student_class  TEXT
      );
      DROP TABLE IF EXISTS courses;
      CREATE TABLE courses (
        course_id       TEXT     PRIMARY KEY,
        course_name     TEXT,
        course_credits  INTEGER
      );
      DROP TABLE IF EXISTS enrolments;
      CREATE TABLE enrolments (
        student_id  TEXT,
        section_id  TEXT,
        course_id   TEXT,
        term_id     TEXT
      );
    """)

# Lưu dữ liệu đăng ký học vào database

In [7]:
def save_to_database(records: list[DkmhUETRecord]) -> None:
  with connect("database.db") as conn:
    conn.executemany(
      "INSERT OR IGNORE INTO students VALUES (?, ?, ?, ?)",
      [(r.student_id, r.student_name, r.birth_date, r.student_class) for r in records]
    )
    conn.executemany(
      "INSERT OR IGNORE INTO courses VALUES (?, ?, ?)",
      [(r.course_id, r.course_name, r.course_credits) for r in records]
    )
    conn.executemany(
      "INSERT INTO enrolments VALUES (?, ?, ?, ?)",
      [(r.student_id, r.section_id, r.course_id, r.term_id) for r in records]
    )

# Main code: Crawl dữ liệu đăng ký học

In [8]:
create_database()
term_ids = ["251", "243", "242", "241"]
records = []
for term_id in tqdm(term_ids):
  records.extend(crawl(term_id))
save_to_database(records)

print(f"Số lượt đăng ký học: {len(records)}")

100%|██████████| 4/4 [03:03<00:00, 45.98s/it]


Số lượt đăng ký học: 179246
